In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [3]:
# Deterministic approach

def deterministic_guardrails(text: str) -> bool:
    """Returns True if the text is appropriate, False otherwise."""
    banned_words = ["hack", "exploit", "malware", "bomb", "terror"]
    return any(kw in text.lower() for kw in banned_words)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

for inp in test_inputs:
    blocked = deterministic_guardrails(inp)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status}: {inp}")

Blocked: How do I hack into a database?
Allowed: What is the capital of France?
Blocked: Explain how malware spreads


In [14]:
# model-based approach

from langchain_google_genai import ChatGoogleGenerativeAI

def model_based_guardrails(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    prompt = f"""Evaluate the following text for safety. Respond with SAFE or UNSAFE.
    \n\nText: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    return result

for inp in test_inputs:
    result = model_based_guardrails(inp)
    status = "SAFE" if result == "SAFE" else "UNSAFE"
    print(f"{status}: {inp}")

UNSAFE: How do I hack into a database?
UNSAFE: What is the capital of France?
UNSAFE: Explain how malware spreads


#### LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def process_user_input(user_input: str) -> str:
    """Look up customer information"""
    return f"customer record found for query: {user_input}"

agent = create_agent(
    tools=[process_user_input],
    model="groq:llama-3.1-8b-instant",
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            strategy="block",
            detector = r"sk-[a-zA-Z0-9]{32}",
            apply_to_input=True,
        )
    ]
)

print("Agent with PII Middleware creating successfully.")

Agent with PII Middleware creating successfully.


In [21]:
result = agent.invoke({
    "messages": [{"role": "user",
                  "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"}]
})

print("Agent_response:", result["messages"][-1].content)

Agent_response: I can't assist with that request. Is there something else I can help you with?


In [22]:
try:
    result = agent.invoke({
        "messages" : [{"role": "user",
                       "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"}]
    })

except Exception as e:
    print("Agent blocked input with API key:", str(e))

Agent blocked input with API key: Detected 1 instance(s) of api_key in text content


# Human in the loop middleware example

### Pauses agent execution before sensitive operations and waits for human approval.

### Best for:

- Financial transactions
- Sending emails to external parties
- Deleting production data
- Any operation with significant business impact

**Key requirement**: A checkpointer for state persistence across interrupts.

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def search_record(query:str)-> str:
    """search the human query on the internet"""
    return f"Search result for {query}"

@tool
def send_email(to: str, subject: str, body: str)->str:
    """send an email to recipient"""
    return f"send an email to {to} with subject {subject} and body {body}"

@tool
def delete_records(table: str, condition: str)-> str:
    """Delete the records from the database"""
    return f"Delete the records from the table {table} where conditions {condition}"

# Create an agent with HITL middleware
hitl_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_record, send_email, delete_records],
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email":True,
            "search_record":False,
            "delete_records": True
        }
    )],
    checkpointer=InMemorySaver(),

)
print("Human in the loop middleware is ready")

Human in the loop middleware is ready


In [41]:
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role":"user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused : waiting for human approval")
print(result)

=== Agent paused : waiting for human approval
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='d67c02ab-a309-4222-aff1-06bf114a825e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'hydzgbvmk', 'function': {'arguments': '{"body":"Please find the Q4 results attached.","subject":"Q4 Results","to":"team@company.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 351, 'total_tokens': 389, 'completion_time': 0.057328202, 'completion_tokens_details': None, 'prompt_time': 0.031247544, 'prompt_tokens_details': None, 'queue_time': 0.049314976, 'total_time': 0.088575746}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9abd-0f92-79f0-88c9-e9185b84fd35-0', tool_cal

In [42]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions":[{"type": "approve"}]}),
    config=config
)
print("===approved===")
print(approved_result["messages"][-1].content)

===approved===



In [43]:
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

=== Rejected! Final response ===



# Custom Guardrail — Before-Agent Hook (Input Filter)

Use before_agent() to validate or block requests before any LLM processing begins.

**Best For**
- Keyword/content filtering
- Authentication checks
- Rate limiting
- Blocking specific categories of requests

In [51]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.tools import tool
from langchain.agents import create_agent

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state:AgentState, runtime:Runtime)-> dict[str, Any] | None:
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return{
                    "messages":[{
                        "role": "assistant",
                        "content":(
                            "I cannot process requests containing inappropriate content."
                            "Please Rephrase your request."
                        )
                    }],
                    "jump_to":"end"
                }
        return None
    
@tool
def search_tool(query:str)->str:
    """Web search for the query"""
    return f"Results for: {query}"

filtered_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)
print("content filter agent created")

content filter agent created


In [52]:
# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
Machine learning is a subset of artificial intelligence (AI) that involves the use of algorithms and statistical models to enable machines to learn from data, make decisions, and improve their performance on a task without being explicitly programmed.


In [53]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content.Please Rephrase your request.


# Custom Guardrail — After-Agent Hook (Output Safety)

Use after_agent() to validate the final agent response before the user sees it.

*Best for*
- Model-based safety evaluation of outputs
- Compliance scanning (e.g. legal, medical, financial disclaimers)
- Quality validation
- Removing sensitive info that slipped through

In [63]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user."""

    def __init__(self):
        super().__init__()
        self.safety_model = ChatGroq(model="groq:llama-3.1-8b-instant",temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state:AgentState, runtime:Runtime)-> dict[str, Any]|None:
        if not state["messages"]:
            return None
        
        last_message = state["messages"][-1].content
        if not isinstance(last_message, AIMessage):
            return None
        
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.   

Response to evaluate: {last_message.content}"""
        result = self.safety_model.invoke([HumanMessage(content=safety_prompt)])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )
        return None
    
@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"

safe_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [64]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [HumanMessage(content="what is the weather in surat today?")]
})
print("Response:")
print(result["messages"][-1].content)

Response:
<brave_search>


# Layered / Combined Guardrails
Stack multiple guardrails in the middleware=[ ] array. They execute in order, building layered protection.

In [67]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full Layered Guardrail stack
production_agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool":True, "search_tool":False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),

)
print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!
